# Estimated Marginal Means (EMMs) in Python

This tutorial demonstrates how to compute estimated marginal means (EMMs) and post hoc contrasts in Python using **EMMEANSpy**, a lightweight implementation inspired by the R **emmeans** package.

The tutorial introduces the core concepts of estimated marginal means using a simple simulated dataset and walks through a complete analysis workflow, including:

* fitting linear and generalized linear models with `statsmodels`;
* computing estimated marginal means for main effects and interactions;
* estimating pairwise and custom contrasts;
* applying multiple-comparison corrections;
* interpreting results on both the link and response scales where appropriate.

The implementation is designed to reproduce the behaviour of R's **emmeans** as closely as possible while remaining fully integrated into the Python scientific ecosystem. It supports both ordinary linear models and generalized linear models, providing estimated marginal means on the response scale and hypothesis tests on the model's native link scale, consistent with the statistical methodology used by **emmeans**.

This notebook is intended for researchers and students who wish to perform reproducible post hoc analyses in Python without relying on R, while obtaining results that closely mirror those produced by the original **emmeans** package.


In [ ]:
# -----------------------------
# 0. Imports
# -----------------------------

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.libqsturng import psturng

from EMMEANSpy import emmeans
from EMMEANSpy import utils


In [2]:
"""
GLMM-style workflow using statsmodels + emmeanspy
"""

# -----------------------------
# 1. Load / simulate data
# -----------------------------

# df = utils.make_example_data(seed=42, n=400)
# df.to_csv("emmeans_tutorial_data.csv", index=False)
df = pd.read_csv("emmeans_tutorial_data.csv")

# Ensure categorical structure (important for EMMs)
df["treatment"] = pd.Categorical(df["treatment"],
                                 categories=["control", "drugA", "drugB"])

df["load"] = pd.Categorical(df["load"],
                            categories=["high_load", "low_load"])

# -----------------------------
# 2. Fit model (GLM / GLMM-style approximation)
# -----------------------------
# Note: statsmodels uses GLM for Gamma; closest analogue to lme4::glmer

model = smf.glm(
    "y ~ treatment * load + age",
    data=df,
    family=sm.families.Gamma(sm.families.links.log())
).fit()

# -----------------------------
# 3. Model summary / ANOVA
# -----------------------------
print(model.summary())

# Type II-like ANOVA table (optional)
anova_table = model.summary()

# -----------------------------
# 4. Estimated marginal means (EMMs)
# -----------------------------

# --- Main effect: treatment ---
emm_treatment = emmeans.emmeans(
    model,
    data=df,
    specs="treatment",
    transform="response",
    contrasts="pairwise",
    adjust="bonferroni",
    contrast_transform="link"
)

# --- Interaction: treatment within load ---
emm_treatment_by_load = emmeans.emmeans(
    model,
    data=df,
    specs="treatment",
    by="load",
    transform="response",
    contrasts="pairwise",
    adjust="bonferroni",
    contrast_transform="link"
)

# --- Reverse slicing: load within treatment ---
emm_load_by_treatment = emmeans.emmeans(
    model,
    data=df,
    specs="load",
    by="treatment",
    transform="response",
    contrasts="pairwise",
    adjust="bonferroni",
    contrast_transform="link"
)

# -----------------------------
# 5. Extract results
# -----------------------------
emm_table = emm_treatment["emm"]
contrasts_table = emm_treatment["contrasts"]

print("\nEMMs:")
print(emm_table)

print("\nContrasts:")
print(contrasts_table)

# -----------------------------
# 6. Save outputs (R-style equivalent)
# -----------------------------
import pickle

outputs = {
    "model": model,
    "anova": anova_table,
    "emm_treatment": emm_treatment,
    "emm_treatment_by_load": emm_treatment_by_load,
    "emm_load_by_treatment": emm_load_by_treatment,
}

with open("posthoc_results.pkl", "wb") as f:
    pickle.dump(outputs, f)


                 Generalized Linear Model Regression Results                  
Dep. Variable:                      y   No. Observations:                  400
Model:                            GLM   Df Residuals:                      393
Model Family:                   Gamma   Df Model:                            6
Link Function:                    log   Scale:                        0.031541
Method:                          IRLS   Log-Likelihood:                -579.17
Date:                Tue, 30 Jun 2026   Deviance:                       13.268
Time:                        17:43:40   Pearson chi2:                     12.4
No. Iterations:                     8   Pseudo R-squ. (CS):             0.6234
Covariance Type:            nonrobust                                         
                                          coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

d:\BonoKat\research project\EMMEANSpy\.venv\Lib\site-packages\statsmodels\genmod\families\links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(
